# Motor-state encoding: paw vs whisk vs lick

Split the motor-STATES group of the encoding model into **paw / whisk / lick** and
compare how much each state uniquely explains of neural activity, across sessions.

Per neuron we compute, controlling for task and the other two states:
- **dR2_<state>**  = unique variance (full - model-without-that-state)
- **r2_<state>_only** = single-variable cvR2 (that state alone)

Fit logic is in `encoding_functions.fit_session_motor_split`; this notebook is the
driver (per-probe cache, resumable, scalable to all sessions).

Note: paw is 7-8 one-hot states, whisk/lick are 1 each — so paw has more DOF; whisk
matching paw with a single regressor is the notable comparison.

In [ ]:
import os, glob, pickle, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from iblatlas.atlas import BrainRegions
import encoding_functions as ef
importlib.reload(ef)

prefix = '/home/ines/repositories/'
neuron_dir = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'

_br = BrainRegions(); _cache = {}
def beryl(a):
    if a in _cache: return _cache[a]
    try: v = _br.id2acronym(_br.remap(_br.acronym2id(a), source_map='Allen', target_map='Beryl'))[0]
    except Exception: v = None
    _cache[a] = v; return v

## Sweep over probes (cached, resumable, scalable to all sessions)

In [ ]:
MOTOR_LAGS = ()                     # () = instantaneous one-hot; ('motor_states',) to lag
results_dir = 'motor_split_results'
os.makedirs(results_dir, exist_ok=True)
pids = [os.path.basename(f).replace('states_neurons_file_', '')
        for f in glob.glob(neuron_dir + 'states_neurons_file_*')]
N_PROBES = len(pids)                # all probes

for i, pid in enumerate(pids[:N_PROBES]):
    cache = os.path.join(results_dir, pid + '.parquet')
    if os.path.exists(cache):
        continue
    try:
        with open(neuron_dir + 'states_neurons_file_' + pid, 'rb') as f:
            df = pickle.load(f)
        r = ef.fit_session_motor_split(df, motor_lags=MOTOR_LAGS)
        r['pid'] = pid; r['session'] = df['session'].iloc[0]; r['mouse_name'] = df['mouse_name'].iloc[0]
        r.to_parquet(cache)
        print(f'[{i+1}/{N_PROBES}] {pid[:8]}: {len(r)} neurons', flush=True)
    except Exception as e:
        print(f'[{i+1}/{N_PROBES}] {pid[:8]} FAIL {type(e).__name__}: {e}', flush=True)
print('cached probes:', len(glob.glob(results_dir + '/*.parquet')))

## Aggregate: paw vs whisk vs lick

In [ ]:
R = pd.concat([pd.read_parquet(f) for f in glob.glob(results_dir + '/*.parquet')], ignore_index=True)
R['beryl'] = R['area'].map({a: beryl(a) for a in R['area'].unique()})
print(f'{len(R)} neurons | {R["pid"].nunique()} probes | {R["session"].nunique()} sessions | {R["mouse_name"].nunique()} mice\n')

STATES = ['paw', 'whisk', 'lick']
print('mean UNIQUE dR2   (single-variable cvR2 in parens):')
for g in ['task'] + STATES:
    if f'dR2_{g}' in R:
        print(f'  {g:6s}: dR2={R[f"dR2_{g}"].mean():.4f}   r2_only={R[f"r2_{g}_only"].mean():.4f}')

# per-neuron winner among the 3 states (by unique dR2)
win = R[[f'dR2_{s}' for s in STATES]].idxmax(axis=1).str.replace('dR2_', '').value_counts()
print('\nper-neuron winner (most unique dR2):'); print(win.to_string())

# per-mouse means, to guard against pooling (paired across states)
pm = R.groupby('mouse_name')[[f'dR2_{s}' for s in STATES]].mean()
print(f'\nmice where whisk>lick: {(pm["dR2_whisk"]>pm["dR2_lick"]).sum()}/{len(pm)} | '
      f'whisk>paw: {(pm["dR2_whisk"]>pm["dR2_paw"]).sum()}/{len(pm)}')

In [ ]:
# Overall bar: unique dR2 (solid) + single-variable cvR2 (faded), per state
fig, ax = plt.subplots(figsize=(6, 4))
colors = {'paw': '#00798c', 'whisk': '#d1495b', 'lick': '#edae49'}
x = np.arange(len(STATES))
ax.bar(x, [R[f'r2_{s}_only'].mean() for s in STATES], color=[colors[s] for s in STATES], alpha=.30)
ax.bar(x, [R[f'dR2_{s}'].mean() for s in STATES], color=[colors[s] for s in STATES])
ax.set_xticks(x); ax.set_xticklabels(STATES)
ax.set(ylabel='variance explained (cvR²)',
       title='Motor states: faded = single-variable cvR², solid = unique ΔR²')
plt.tight_layout(); plt.show()

In [ ]:
# Per-area (Beryl) unique dR2 for paw/whisk/lick (areas with >=30 neurons)
counts = R.groupby('beryl').size()
keep_areas = counts[counts >= 30].index
sub = R[R['beryl'].isin(keep_areas)]
agg = sub.groupby('beryl')[[f'dR2_{s}' for s in STATES]].mean()
agg = agg.loc[agg.sum(1).sort_values(ascending=False).index]
xa = np.arange(len(agg)); w = 0.8 / len(STATES)
fig, ax = plt.subplots(figsize=(13, 4.5))
for i, s in enumerate(STATES):
    ax.bar(xa + (i - 1) * w, agg[f'dR2_{s}'], w, label=s, color=colors[s])
ax.set_xticks(xa); ax.set_xticklabels(agg.index, rotation=45, ha='right')
ax.set(ylabel='unique ΔR²', title='paw vs whisk vs lick — unique variance per area (≥30 neurons)')
ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()
agg.round(4)